<a href="https://colab.research.google.com/github/vidalzfelipe/ClearBank-analise-financeira-com-python/blob/main/ClearBank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Leitura do Arquivo CSV com Módulo Nativo

In [88]:
import csv

try:
    with open("transacoes.csv", "r", encoding="utf-8") as arquivo:
        leitor = csv.DictReader(arquivo)

        transacoes_brutas = list(leitor)

        for linha in transacoes_brutas:
            print(linha)

except FileNotFoundError:
    print("Arquivo não encontrado.")

{'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salario janeiro', 'categoria': 'salario'}
{'id': '2', 'data': '2026-01-08', 'cliente_id': 'CLI002', 'tipo': 'debito', 'valor': '245.90', 'descricao': 'Supermercado', 'categoria': 'compra'}
{'id': '3', 'data': '2026-01-12', 'cliente_id': 'CLI003', 'tipo': 'debito', 'valor': '89.90', 'descricao': 'Farmacia', 'categoria': 'saude'}
{'id': '4', 'data': '2026-01-18', 'cliente_id': 'CLI004', 'tipo': 'credito', 'valor': '12500.00', 'descricao': 'Venda de equipamento', 'categoria': 'venda'}
{'id': '5', 'data': '2026-01-25', 'cliente_id': 'CLI005', 'tipo': 'debito', 'valor': '420.00', 'descricao': 'Conta de energia', 'categoria': 'conta'}
{'id': '6', 'data': '2026-02-03', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salario fevereiro', 'categoria': 'salario'}
{'id': '7', 'data': '2026-02-07', 'cliente_id': 'CLI002', 'tipo': 'debito', 'valor': '180.50', 'd

## Validação e limpeza dos dados

In [89]:
from datetime import datetime

transacoes_validas = []

total_linhas = 0
linhas_validas = 0
linhas_invalidas = 0

for linha in transacoes_brutas:
    total_linhas += 1

    try:
        # id
        if linha["id"].strip() == "" or not linha["id"].strip().isdigit():
            raise ValueError

        # cliente_id
        if linha["cliente_id"].strip() == "":
            raise ValueError

        # data
        datetime.strptime(linha["data"], "%Y-%m-%d")

        # tipo
        if linha["tipo"] not in ["credito", "debito"]:
            raise ValueError

        # valor
        valor = float(linha["valor"])

        if valor <= 0:
            raise ValueError

        # Transação Válida
        transacoes_validas.append({
            "id": int(linha["id"]),
            "data": linha["data"],
            "cliente_id": linha["cliente_id"],
            "tipo": linha["tipo"],
            "valor": valor,
            "descricao": linha["descricao"],
            "categoria": linha["categoria"]
        })

        linhas_validas += 1

    except (ValueError, KeyError):
        linhas_invalidas += 1

print(f"Total de linhas lidas: {total_linhas}")
print(f"Linhas válidas: {linhas_validas}")
print(f"Linhas inválidas: {linhas_invalidas}")

Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5


## Organização do Código em Funções

### ler_transacoes()

In [90]:
def ler_transacoes(nome_arquivo):
    try:
        with open(nome_arquivo, "r", encoding="utf-8") as arquivo:
            return list(csv.DictReader(arquivo))
    except FileNotFoundError:
        print("Arquivo não encontrado.")
        return []


### validar_transacao()

In [91]:
def validar_transacao(transacoes_brutas):
    transacoes_validas = []

    total_linhas = 0
    linhas_validas = 0
    linhas_invalidas = 0

    for linha in transacoes_brutas:
        total_linhas += 1

        try:
            # id
            if linha["id"].strip() == "" or not linha["id"].strip().isdigit():
                raise ValueError

            # cliente_id
            if linha["cliente_id"].strip() == "":
                raise ValueError

            # data
            data_obj = datetime.strptime(linha["data"], "%Y-%m-%d")

            # tipo
            if linha["tipo"] not in ["credito", "debito"]:
                raise ValueError

            # valor
            valor = float(linha["valor"])

            if valor <= 0:
                raise ValueError

            # Transação válida
            transacoes_validas.append({
                "id": int(linha["id"]),
                "data": linha["data"],
                "data_obj": data_obj,
                "mes": data_obj.strftime("%Y-%m"),
                "cliente_id": linha["cliente_id"],
                "tipo": linha["tipo"],
                "valor": valor,
                "descricao": linha["descricao"],
                "categoria": linha["categoria"]
            })

            linhas_validas += 1

        except (ValueError, KeyError):
            linhas_invalidas += 1

    return transacoes_validas, total_linhas, linhas_validas, linhas_invalidas

### gerar_relatorio()

In [92]:
LIMITE_SUSPEITO = 10000.00

def gerar_relatorio(transacoes):
    metricas = {}
    suspeitas = []

    datas = [transacao["data_obj"] for transacao in transacoes]

    data_mais_antiga = min(datas)
    data_mais_recente = max(datas)
    dias_entre = (data_mais_recente - data_mais_antiga).days

    for transacao in transacoes:
        mes = transacao["mes"]

        # Identifica transações suspeitas
        if transacao["valor"] > LIMITE_SUSPEITO:
            suspeitas.append(transacao)

        if mes not in metricas:
            metricas[mes] = {
                "quantidade_total": 0,
                "total_credito": 0,
                "total_debito": 0,
                "saldo_mes": 0,
                "valor_medio": 0,
                "maior_transacao": transacao,
                "menor_transacao": transacao
            }

        metricas[mes]["quantidade_total"] += 1

        if transacao["tipo"] == "credito":
            metricas[mes]["total_credito"] += transacao["valor"]

        elif transacao["tipo"] == "debito":
            metricas[mes]["total_debito"] += transacao["valor"]

        if transacao["valor"] > metricas[mes]["maior_transacao"]["valor"]:
            metricas[mes]["maior_transacao"] = transacao

        if transacao["valor"] < metricas[mes]["menor_transacao"]["valor"]:
            metricas[mes]["menor_transacao"] = transacao

    for mes in metricas:
        total_credito = metricas[mes]["total_credito"]
        total_debito = metricas[mes]["total_debito"]
        quantidade = metricas[mes]["quantidade_total"]

        metricas[mes]["saldo_mes"] = total_credito - total_debito
        metricas[mes]["valor_medio"] = (total_credito + total_debito) / quantidade

    return {
        "data_mais_antiga": data_mais_antiga.strftime("%Y-%m-%d"),
        "data_mais_recente": data_mais_recente.strftime("%Y-%m-%d"),
        "dias_entre_transacoes": dias_entre,
        "metricas_por_mes": metricas,
        "transacoes_suspeitas": suspeitas
    }

### salvar_json()

In [93]:
import json
from datetime import datetime


def salvar_json(metricas, linhas_validas, linhas_invalidas):
    relatorio = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": linhas_validas,
        "total_transacoes_invalidas": linhas_invalidas,
        "resumo_mensal": {}
    }

    for mes, dados in metricas["metricas_por_mes"].items():
        relatorio["resumo_mensal"][mes] = {
            "quantidade": dados["quantidade_total"],
            "total_credito": dados["total_credito"],
            "total_debito": dados["total_debito"],
            "saldo": dados["saldo_mes"],
            "media": dados["valor_medio"],
            "maior_valor": dados["maior_transacao"]["valor"],
            "menor_valor": dados["menor_transacao"]["valor"]
        }

    with open("relatorio.json", "w", encoding="utf-8") as arquivo:
        json.dump(relatorio, arquivo, indent=4, ensure_ascii=False)

    print("Relatório salvo em relatorio.json")


### exibir_relatorio()

In [94]:
def formatar_real(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(metricas, linhas_validas, linhas_invalidas):
    print("===== RELATÓRIO GERAL =====")
    print(f"Período analisado: {metricas['data_mais_antiga']} → {metricas['data_mais_recente']}")
    print(f"Transações válidas: {linhas_validas}")
    print(f"Transações inválidas: {linhas_invalidas}")
    print()

    print("===== RELATÓRIO MENSAL =====\n")

    for mes, dados in metricas["metricas_por_mes"].items():
        print(f"Mês: {mes}")
        print(f"  Transações: {dados['quantidade_total']}")
        print(f"  Total crédito: {formatar_real(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_real(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_real(dados['saldo_mes'])}")
        print(f"  Média:         {formatar_real(dados['valor_medio'])}")
        print(f"  Maior valor:   {formatar_real(dados['maior_transacao']['valor'])}")
        print(f"  Menor valor:   {formatar_real(dados['menor_transacao']['valor'])}")
        print()

    print("\n===== TRANSAÇÕES SUSPEITAS =====")

    suspeitas = metricas["transacoes_suspeitas"]

    if len(suspeitas) == 0:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for transacao in suspeitas:
            print(
                f"ID: {transacao['id']} | "
                f"Cliente: {transacao['cliente_id']} | "
                f"Data: {transacao['data']} | "
                f"Valor: {formatar_real(transacao['valor'])}"
            )

transacoes_brutas = ler_transacoes("transacoes.csv")

transacoes_validas, total_linhas, linhas_validas, linhas_invalidas = validar_transacao(transacoes_brutas)

metricas = gerar_relatorio(transacoes_validas)

salvar_json(metricas, linhas_validas, linhas_invalidas)

exibir_relatorio(metricas, linhas_validas, linhas_invalidas)

Relatório salvo em relatorio.json
===== RELATÓRIO GERAL =====
Período analisado: 2026-01-05 → 2026-03-26
Transações válidas: 15
Transações inválidas: 5

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 5
  Total crédito: R$ 16.000,00
  Total débito:  R$ 755,80
  Saldo:         R$ 15.244,20
  Média:         R$ 3.351,16
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 89,90

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 21.500,00
  Total débito:  R$ 405,50
  Saldo:         R$ 21.094,50
  Média:         R$ 4.381,10
  Maior valor:   R$ 18.000,00
  Menor valor:   R$ 75,00

Mês: 2026-03
  Transações: 5
  Total crédito: R$ 5.700,00
  Total débito:  R$ 1.060,65
  Saldo:         R$ 4.639,35
  Média:         R$ 1.352,13
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 99,90


===== TRANSAÇÕES SUSPEITAS =====
ID: 4 | Cliente: CLI004 | Data: 2026-01-18 | Valor: R$ 12.500,00
ID: 9 | Cliente: CLI004 | Data: 2026-02-16 | Valor: R$ 18.000,00
